In [1]:
import pandas as pd

train_df = pd.read_csv('/kaggle/input/foodvqa/train.csv')
test_df = pd.read_csv('/kaggle/input/foodvqa/test.csv')
val_df = pd.read_csv('/kaggle/input/foodvqa/validation.csv')

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Extract visual feature

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.ops import roi_align
from torchvision.transforms.functional import resize, to_tensor
from PIL import Image
import numpy as np

model_rcnn = fasterrcnn_resnet50_fpn(pretrained=True).eval().to(device)
for param in model.parameters():
    param.requires_grad = False

def extract_visual_feats(image_path, max_regions=36, target_size=480):
    image = Image.open(image_path).convert("RGB")
    image = resize(image, [target_size, target_size])
    img_tensor = to_tensor(image).unsqueeze(0).to(device)  # [1, 3, H, W]

    with torch.no_grad():
        outputs = model_rcnn(img_tensor)

    boxes = outputs[0]['boxes'][:max_regions]
    if len(boxes) == 0:
        return None, None

    # Extract feature map
    features = model_rcnn.backbone(img_tensor)
    feature_map = features['0']  # [1, 256, H', W']

    # ROI Align
    batch_idx = torch.zeros((boxes.shape[0], 1), device=device)
    rois = torch.cat([batch_idx, boxes], dim=1)
    aligned = roi_align(feature_map, rois, output_size=(7, 7))  # [N, 256, 7, 7]

    # Flatten theo đúng input của box_head (36, 12544)
    flattened = aligned.view(aligned.size(0), -1)

    # Trích visual_feats chuẩn từ box_head
    box_head = model_rcnn.roi_heads.box_head
    visual_feats_1024 = box_head(flattened)
    visual_feats = torch.nn.functional.pad(visual_feats_1024, (0, 1024))
    
    return visual_feats, boxes

# visual_feats, visual_pos = extract_visual_feats(image_path)

# print("visual_feats shape:", visual_feats.shape)  # [N, 2048]
# print("visual_pos shape:", visual_pos.shape)      # [N, 4]

In [ ]:
import os

def create_id_to_filepath(df, assets_path):
    extensions = ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']
    
    # Tạo dictionary ID to file path
    id_to_filepath = {}
    for img_id in df['Image'].unique():
        for ext in extensions:
            file_path = os.path.join(assets_path, img_id + ext)
            if os.path.exists(file_path):
                id_to_filepath[img_id] = file_path
                break
    
    return id_to_filepath

In [ ]:
import pandas as pd
from tqdm import tqdm

visual_feats_list = []
visual_pos_list = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Extracting visual features"):
    image_path = id2filename[row["Image"]]

    try:
        feats, pos = extract_visual_feats(image_path)
    except Exception as e:
        print(f"Error image: {image_path}, skip because: {e}")
        feats, pos = None, None

    visual_feats_list.append(feats.detach().cpu().numpy() if feats is not None else None)
    visual_pos_list.append(pos.detach().cpu().numpy() if pos is not None else None)

    torch.cuda.empty_cache()
    
test_visual_df = test_df.copy()
test_visual_df["visual_feats"] = visual_feats_list
test_visual_df["visual_pos"] = visual_pos_list

# LXMERT pipeline

In [3]:
from transformers import LxmertTokenizer, LxmertForQuestionAnswering

tokenizer = LxmertTokenizer.from_pretrained("unc-nlp/lxmert-vqa-uncased")
model = LxmertForQuestionAnswering.from_pretrained("unc-nlp/lxmert-vqa-uncased").to(device)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/856M [00:00<?, ?B/s]

In [4]:
from huggingface_hub import snapshot_download
import pandas as pd
import os
import shutil

repo_path = snapshot_download(repo_id="huyg1108/beit3-foodvqa-lxmert", repo_type="model")

for fname in ['train_visual.pkl', 'val_visual.pkl']:
    src = os.path.join(repo_path, fname)
    dst = os.path.join('/kaggle/working', fname)
    shutil.copy(src, dst)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

val_visual.pkl:   0%|          | 0.00/778M [00:00<?, ?B/s]

train_visual.pkl:   0%|          | 0.00/6.34G [00:00<?, ?B/s]

val_visual.csv:   0%|          | 0.00/283k [00:00<?, ?B/s]

train_visual.csv:   0%|          | 0.00/2.27M [00:00<?, ?B/s]

## Load dataset (including visual features)

In [5]:
train_df = pd.read_pickle('/kaggle/working/train_visual.pkl')
val_df = pd.read_pickle('/kaggle/working/val_visual.pkl')

In [6]:
train_df["Answer"] = train_df["Answer"].str.lower()
val_df["Answer"] = val_df["Answer"].str.lower()
test_df["Answer"] = test_df["Answer"].str.lower()

In [7]:
train_df

,Image,Question,Answer,visual_feats,visual_pos
0,2e789c07ec72245,What color is the ground beef inside the enchi...,reddish-brown,"[[0.03210245, 0.013213702, 0.14927956, 0.0, 0....","[[15.337811, 0.48733523, 480.00003, 421.81693]..."
1,2e789c07ec72245,What type of dish is shown in the image,enchiladas,"[[0.03210245, 0.013213702, 0.14927956, 0.0, 0....","[[15.337811, 0.48733523, 480.00003, 421.81693]..."
2,2e789c07ec72245,What color is the rice next to the enchiladas,orange,"[[0.03210245, 0.013213702, 0.14927956, 0.0, 0....","[[15.337811, 0.48733523, 480.00003, 421.81693]..."
3,2e789c07ec72245,Where are the sliced black olives placed,on top,"[[0.03210245, 0.013213702, 0.14927956, 0.0, 0....","[[15.337811, 0.48733523, 480.00003, 421.81693]..."
4,2e789c07ec72245,What is the enchilada sauce and melted cheese ...,plate,"[[0.03210245, 0.013213702, 0.14927956, 0.0, 0....","[[15.337811, 0.48733523, 480.00003, 421.81693]..."
...,...,...,...,...,...
34709,69fc8de514a3aba,What color is the bottom layer of the dessert,yellow,"[[0.0, 0.0, 0.1478942, 0.0, 0.06121485, 0.0846...","[[1.2181091, 45.042793, 443.40396, 364.1432], ..."
34710,69fc8de514a3aba,What type of dish is Zuppa Inglese,dessert,"[[0.0, 0.0, 0.1478942, 0.0, 0.06121485, 0.0846...","[[1.2181091, 45.042793, 443.40396, 364.1432], ..."
34711,69fc8de514a3aba,What is the state of the sponge cake in the de...,soaked,"[[0.0, 0.0, 0.1478942, 0.0, 0.06121485, 0.0846...","[[1.2181091, 45.042793, 443.40396, 364.1432], ..."
34712,69fc8de514a3aba,What color is the chocolate pudding layer,brown,"[[0.0, 0.0, 0.1478942, 0.0, 0.06121485, 0.0846...","[[1.2181091, 45.042793, 443.40396, 364.1432], ..."


In [8]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class VQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.label2id = label2id

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = row["Question"]
        answer = self.label2id[row["Answer"]]

        encoded = tokenizer(
            question,
            padding="max_length",
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "visual_feats": torch.tensor(row["visual_feats"], dtype=torch.float),
            "visual_pos": torch.tensor(row["visual_pos"], dtype=torch.float),
            "label": torch.tensor(answer, dtype=torch.long)
        }

In [9]:
import torch.nn.functional as F

def custom_collate(batch, max_regions=36):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["label"] for item in batch])

    def pad_tensor(tensor, max_len, pad_value=0.0):
        # tensor shape: [N, dim]
        if tensor.size(0) >= max_len:
            return tensor[:max_len]
        pad_size = (0, 0, 0, max_len - tensor.size(0))  # (dim2_pad, dim1_pad)
        return F.pad(tensor, pad_size, value=pad_value)

    visual_feats = torch.stack([pad_tensor(item["visual_feats"], max_regions) for item in batch])
    visual_pos = torch.stack([pad_tensor(item["visual_pos"], max_regions) for item in batch])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "visual_feats": visual_feats,
        "visual_pos": visual_pos,
        "label": labels
    }

In [10]:
from sklearn.preprocessing import LabelEncoder

all_answers = pd.concat([
    train_df["Answer"],
    val_df["Answer"],
    test_df["Answer"]
], ignore_index=True)

label_encoder = LabelEncoder()
label_encoder.fit(all_answers)

label2id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id2label = {idx: label for label, idx in label2id.items()}

In [11]:
train_dataset = VQADataset(train_df, tokenizer, label2id)
val_dataset = VQADataset(val_df, tokenizer, label2id)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=lambda x: custom_collate(x, max_regions=36))
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=lambda x: custom_collate(x, max_regions=36))

## Training model

In [12]:
%%capture
import torch.nn as nn
model.num_qa_labels = len(id2label)
model.answer_head.logit_fc[-1] = nn.Linear(in_features=1536, out_features=len(label2id), bias=True)
model.to(device)

In [13]:
from tqdm import tqdm
import torch

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
EPOCHS = 100
best_val_loss = float("inf")
epochs_no_improve = 0
patience = 3
early_stop = False

for epoch in range(EPOCHS):
    if early_stop:
        print("Early stopping triggered.")
        break

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        visual_feats = batch["visual_feats"].to(device)
        visual_pos = batch["visual_pos"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            visual_feats=visual_feats,
            visual_pos=visual_pos,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}")

    # Validation
    model.eval()
    val_loss_total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual_feats = batch["visual_feats"].to(device)
            visual_pos = batch["visual_pos"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_feats=visual_feats,
                visual_pos=visual_pos,
                labels=labels
            )

            val_loss_total += outputs.loss.item()

    avg_val_loss = val_loss_total / len(val_loader)
    print(f"Epoch {epoch+1}: Val Loss = {avg_val_loss:.4f}")

    # Early stopping based on val loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/best_lxmert_model.pth")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")
        if epochs_no_improve >= patience:
            early_stop = True

Epoch 1: 100%|██████████| 4340/4340 [10:25<00:00,  6.94it/s]


Epoch 1: Train Loss = 4.3858
Epoch 1: Val Loss = 3.9098


Epoch 2: 100%|██████████| 4340/4340 [10:25<00:00,  6.94it/s]


Epoch 2: Train Loss = 3.4290
Epoch 2: Val Loss = 3.8349


Epoch 3: 100%|██████████| 4340/4340 [10:25<00:00,  6.93it/s]


...

Epoch 4: Train Loss = 2.6340
Epoch 4: Val Loss = 3.9166
No improvement for 1 epoch(s)


Epoch 5: 100%|██████████| 4340/4340 [10:31<00:00,  6.88it/s]


Epoch 5: Train Loss = 2.2844
Epoch 5: Val Loss = 4.0326
No improvement for 2 epoch(s)


Epoch 6: 100%|██████████| 4340/4340 [10:32<00:00,  6.87it/s]


Epoch 6: Train Loss = 1.9704
Epoch 6: Val Loss = 4.1343
No improvement for 3 epoch(s)
Early stopping triggered.


# Inference

In [23]:
assets_path = '/kaggle/input/foodvqa/assets'
id2filename_train = create_id_to_filepath(train_df, assets_path)
id2filename_val = create_id_to_filepath(val_df, assets_path)
id2filename_test = create_id_to_filepath(test_df, assets_path)
id2filename = id2filename_train | id2filename_val | id2filename_test

In [22]:
model.load_state_dict(torch.load('/kaggle/working/best_lxmert_model.pth', weights_only=True))

<All keys matched successfully>

In [38]:
import torch
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import LxmertTokenizer
import time

def evaluate_model(model, df, image_col, question_col, answer_col, device, config):
    model.eval()
    correct_predictions = 0
    total_predictions = 0
    all_true_answers = []
    all_predicted_answers = []
    
    with torch.no_grad(): 
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
            image_path = id2filename[str(row[image_col])]
            question = row[question_col]
            true_answer = row[answer_col]
            start_time = time.time()

            # visual features
            feats, pos = extract_visual_feats(image_path)
            if feats is None or pos is None:
                continue

            feats = feats[:36]
            pos = pos[:36]

            # Pad if < 36
            if feats.size(0) < 36:
                pad_len = 36 - feats.size(0)
                feats = torch.cat([feats, torch.zeros((pad_len, 2048)).to(device)], dim=0)
                pos = torch.cat([pos, torch.zeros((pad_len, 4)).to(device)], dim=0)

            # Tokenize question
            inputs = tokenizer(question, return_tensors="pt").to(device)

            inputs.update({
                "visual_feats": feats.unsqueeze(0),   # [1, 36, 2048]
                "visual_pos": pos.unsqueeze(0),       # [1, 36, 4]
            })
            

            # Inference
            outputs = model(**inputs)

            end_time = time.time()
            first_time = end_time - start_time
            print(f"⏱️ Time for first query: {first_time:.4f} seconds")
            
            logits = outputs.question_answering_score  # [1, num_labels]
            predicted_class_id = torch.argmax(logits, dim=1).item()
            predicted_answer = config.get(predicted_class_id, None)

            all_true_answers.append(true_answer)
            all_predicted_answers.append(predicted_answer)

            if predicted_answer is not None and predicted_answer == true_answer:
                correct_predictions += 1
            total_predictions += 1

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    precision = precision_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)
    recall = recall_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)
    f1 = f1_score(all_true_answers, all_predicted_answers, average='weighted', zero_division=1)

    return accuracy, precision, recall, f1, all_true_answers, all_predicted_answers

In [39]:
accuracy, precision, recall, f1, all_true_answers, all_predicted_answers = evaluate_model(
    model, test_df, image_col="Image", question_col="Question", answer_col="Answer", 
    device=device, config=id2label
)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Evaluating:   0%|          | 0/3699 [00:00<?, ?it/s]

⏱️ Time for first query: 0.1307 seconds
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000


In [33]:
import pandas as pd

results_df = pd.DataFrame({
    "Answer": all_true_answers,
    "Predict": all_predicted_answers
})
results_df

,Answer,Predict
0,golden-brown,golden-brown
1,enchiladas,salad
2,white,yellow
3,on the left side,in the center
4,tomatoes,zucchini
...,...,...
3694,throughout the entire pan,scattered across
3695,yellow,green
3696,zucchini,zucchini
3697,black,red


## Save results

In [35]:
results_df.to_csv('/kaggle/working/lxmert_results.csv',index=False)